# PLR Histogram: Part Load Ratio Frequency Distribution

Dual setpoint (cooling 26°C, heating 20°C). Data: `dual_setpoint_small_office_hour.csv`.
This notebook produces only the PLR histogram figure (cooling and heating panels).

## Import library

In [1]:
# Import required libraries
import sys
sys.path.append('src')
import enex_analysis as enex
import matplotlib.pyplot as plt
import dartwork_mpl as dm
import numpy as np
import pandas as pd
from matplotlib.ticker import AutoMinorLocator
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


## Graph Setting

In [2]:
# Plot style
plt.rcParams['font.size'] = 9
fs = {
    'label': dm.fs(0),
    'tick': dm.fs(-1.0),
    'legend': dm.fs(-1.5),
    'subtitle': dm.fs(-0.5),
}
pad = {'label': 6, 'tick': 5}
dm.style.use('scientific')
print("Plot style configured successfully.")

Plot style configured successfully.


## Load data setting

In [3]:
# Load data (dual setpoint: cooling 26°C, heating 20°C)
data_file = '../data/dual_setpoint_small_office_hour.csv'
weekday_df = pd.read_csv(data_file)

cols = list(weekday_df.columns)
date_col = None
heating_col = None
cooling_col = None

for c in cols:
    if c.strip() == 'Date/Time' or 'Date/Time' in c:
        date_col = c
        break
for c in cols:
    if 'DistrictHeatingWater:Facility' in c:
        heating_col = c
        break
for c in cols:
    if 'DistrictCooling:Facility' in c:
        cooling_col = c
        break

weekday_df['Date/Time_clean'] = weekday_df[date_col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
weekday_df['Relative_Day_Index'] = weekday_df.index // 24
weekday_df['Day'] = (6 + weekday_df['Relative_Day_Index']) % 7

if cooling_col is not None:
    weekday_df.loc[weekday_df['Day'].isin([5, 6]), cooling_col] = 0
if heating_col is not None:
    weekday_df.loc[weekday_df['Day'].isin([5, 6]), heating_col] = 0

weekday_df['Month'] = weekday_df['Date/Time_clean'].str.slice(0, 2).astype(int)
if cooling_col is not None:
    weekday_df.loc[weekday_df['Month'].isin([1, 2, 3, 10, 11, 12]), cooling_col] = 0
if heating_col is not None:
    weekday_df.loc[weekday_df['Month'].isin([4, 5, 6, 7, 8, 9]), heating_col] = 0

weekday_df.reset_index(drop=True, inplace=True)
if heating_col is not None:
    weekday_df[heating_col] = pd.to_numeric(weekday_df[heating_col], errors='coerce').fillna(0.0)
if cooling_col is not None:
    weekday_df[cooling_col] = pd.to_numeric(weekday_df[cooling_col], errors='coerce').fillna(0.0)

print(f"Data loaded successfully. Shape: {weekday_df.shape}")

Data loaded successfully. Shape: (8760, 19)


# PLR 상하한 포함 실제 부하 데이터
PLR 0.2-1.0 내에 있는 데이터만 포함

In [11]:
# PLR Histogram Analysis: Part Load Ratio Frequency Distribution

# System maximum capacity [W]
Q_max = 21000  # Maximum system capacity [W]

# Convert loads from J to W (using s2h conversion factor)
cooling_loads_W = weekday_df[cooling_col] * enex.s2h
heating_loads_W = weekday_df[heating_col] * enex.s2h

# Calculate Part Load Ratio (PLR) for each mode
cooling_PLR = cooling_loads_W / Q_max
heating_PLR = heating_loads_W / Q_max

# Filter data: PLR >= 0.2 (exclude PLR < 0.2)
cooling_PLR_filtered = cooling_PLR[(cooling_PLR >= 0.2) & (cooling_PLR <= 1.0)]
heating_PLR_filtered = heating_PLR[(heating_PLR >= 0.2) & (heating_PLR <= 1.0)]

# Define bin edges: 0.2 to 1.0 with 0.05 step size
bin_edges = np.arange(0.2, 1.05, 0.05)

# Reference PLR value for sensitivity analysis
REF_PLR = 0.57

# Create figure with 1 row, 2 columns
fig, (ax_cooling, ax_heating) = plt.subplots(1, 2, figsize=(dm.cm2in(17), dm.cm2in(6)))
plt.subplots_adjust(left=0.06, right=0.975, top=0.93, bottom=0.18, wspace=0.25)

# Define colors
color_cooling = 'oc.blue6'  # Blue for cooling
color_heating = 'oc.red6'   # Red for heating
color_reference = 'oc.red5'  # Red for reference line

# Common settings
alpha_hist = 0.7
line_width = 0.8

# --- Left panel: Cooling Mode PLR Histogram ---
n_cooling, bins_cooling, patches_cooling = ax_cooling.hist(
    cooling_PLR_filtered, bins=bin_edges, density=True, 
    color=color_cooling, alpha=alpha_hist, edgecolor='white', linewidth=0.5
)

# Add reference line at PLR = 0.57 (no legend entry)
ax_cooling.axvline(x=REF_PLR, color=color_reference, linestyle='--', 
                   linewidth=line_width, alpha=0.8)

# Configure axes
ax_cooling.set_xlabel('Part load ratio [ - ]', fontsize=fs['label'], labelpad=pad['label'])
ax_cooling.set_ylabel('Probability density [ - ]', fontsize=fs['label'], labelpad=pad['label'])
ax_cooling.tick_params(axis='both', which='major', labelsize=fs['tick'], pad=pad['tick'])
ax_cooling.set_xlim(0.2, 1.0)
ax_cooling.set_ylim(0, 5)
ax_cooling.set_xticks(np.arange(0.2, 1.1, 0.2))  # Major ticks every 0.2
ax_cooling.xaxis.set_minor_locator(AutoMinorLocator(2))
ax_cooling.yaxis.set_minor_locator(AutoMinorLocator(2))
ax_cooling.grid(True, linestyle='--', alpha=0.6, zorder=0)

# Add panel label (top of axes box)
ax_cooling.text(0.01, 1.02, 'a', transform=ax_cooling.transAxes, 
                fontsize=fs['subtitle'], fontweight='black', va='bottom', ha='left')

# Add legend
ax_cooling.legend(
    loc='upper right', frameon=False,
    fontsize=fs['legend'], fancybox=False,
    bbox_to_anchor=(0.99, 0.99),
    handlelength=1.8
)

# Calculate and display statistics
median_cooling = np.median(cooling_PLR_filtered)
mode_idx_cooling = np.argmax(n_cooling)
mode_cooling = (bins_cooling[mode_idx_cooling] + bins_cooling[mode_idx_cooling + 1]) / 2

# --- Right panel: Heating Mode PLR Histogram ---
n_heating, bins_heating, patches_heating = ax_heating.hist(
    heating_PLR_filtered, bins=bin_edges, density=True,
    color=color_heating, alpha=alpha_hist, edgecolor='white', linewidth=0.5
)

# Add reference line at PLR = 0.57 (no legend entry)
ax_heating.axvline(x=REF_PLR, color=color_reference, linestyle='--', 
                   linewidth=line_width, alpha=0.8)

# Configure axes
ax_heating.set_xlabel('Part load ratio [ - ]', fontsize=fs['label'], labelpad=pad['label'])
ax_heating.set_ylabel('Probability density [ - ]', fontsize=fs['label'], labelpad=pad['label'])
ax_heating.tick_params(axis='both', which='major', labelsize=fs['tick'], pad=pad['tick'])
ax_heating.set_xlim(0.2, 1.0)
ax_heating.set_ylim(0, 5)
ax_heating.set_xticks(np.arange(0.2, 1.1, 0.2))  # Major ticks every 0.2
ax_heating.xaxis.set_minor_locator(AutoMinorLocator(2))
ax_heating.yaxis.set_minor_locator(AutoMinorLocator(2))
ax_heating.grid(True, linestyle='--', alpha=0.6, zorder=0)

# Add panel label (top of axes box)
ax_heating.text(0.01, 1.02, 'b', transform=ax_heating.transAxes, 
                fontsize=fs['subtitle'], fontweight='black', va='bottom', ha='left')

# Add legend
ax_heating.legend(
    loc='upper right', frameon=False,
    fontsize=fs['legend'], fancybox=False,
    bbox_to_anchor=(0.99, 0.99),
    handlelength=1.8
)

# Calculate and display statistics
median_heating = np.median(heating_PLR_filtered)
mode_idx_heating = np.argmax(n_heating)
mode_heating = (bins_heating[mode_idx_heating] + bins_heating[mode_idx_heating + 1]) / 2

# Print statistics
print(f"Cooling Mode - Median PLR: {median_cooling:.3f}, Mode PLR: {mode_cooling:.3f}")
print(f"Heating Mode - Median PLR: {median_heating:.3f}, Mode PLR: {mode_heating:.3f}")

# Save the figure in multiple formats
# plt.savefig('../figure/PLR_histogram.png', dpi=600)
# plt.savefig('../figure/PLR_histogram.pdf', dpi=600)
# plt.savefig('../figure/PLR_histogram.svg', dpi=600, transparent=True)
dm.util.save_and_show(fig)

Cooling Mode - Median PLR: 0.423, Mode PLR: 0.225
Heating Mode - Median PLR: 0.326, Mode PLR: 0.225


# PLR 상하한 제한 설정
PLR 0.2 미만은 0.2로, PLR 1.0 초과는 1.0으로 수정한 데이터

In [23]:
# =============================================================================
# PLR 두 버전 비교: 실제 부하 전체 범위(0~0.2, 1.0 초과 포함) vs 시스템용 0.2~1.0
# =============================================================================
# 버전 1: 실제 데이터 기반 PLR 전체 (0, 0~0.2, 0.2~1.0, 1.0 초과 모두 포함)
# 버전 2: 시스템에서 사용하는 구간만 (0.2 <= PLR <= 1.0)

# Full-range bins (0 to 1.1 so that >1.0 is included)
bin_edges_full = np.arange(0, 1.15, 0.05)  # 0, 0.05, ..., 1.1
# System range bins (same as main figure)
bin_edges_system = np.arange(0.2, 1.05, 0.05)

# PLR 0.2 미만 → 0.2, 1.0 초과 → 1.0 으로 클리핑한 데이터만 사용
cooling_PLR_clipped = cooling_PLR.clip(lower=0.2, upper=1.0)
heating_PLR_clipped = heating_PLR.clip(lower=0.2, upper=1.0)

fig2, axes = plt.subplots(2, 1, figsize=(dm.cm2in(17), dm.cm2in(10)))
plt.subplots_adjust(left=0.08, right=0.975, top=0.92, bottom=0.12, hspace=0.6)

ax_cool_full, ax_heat_full = axes[0], axes[1]

color_cooling = 'oc.blue6'
color_heating = 'oc.red6'
alpha_hist = 0.7

# --- Row 0: Cooling ---
# Left: Full range (actual load data, all PLR)
ax_cool_full.hist(cooling_PLR_clipped, bins=bin_edges_full, density=True,
                  color=color_cooling, alpha=alpha_hist, edgecolor='white', linewidth=0.5)
ax_cool_full.set_xlabel('Part load ratio [ - ]', fontsize=fs['label'], labelpad=pad['label'])
ax_cool_full.set_ylabel('Probability density [ - ]', fontsize=fs['label'], labelpad=pad['label'])
ax_cool_full.tick_params(axis='both', which='major', labelsize=fs['tick'], pad=pad['tick'])
ax_cool_full.set_xlim(0, 1.15)
ax_cool_full.set_ylim(0, 20)
ax_cool_full.set_xticks(np.arange(0, 1.2, 0.2))
ax_cool_full.xaxis.set_minor_locator(AutoMinorLocator(2))
ax_cool_full.yaxis.set_minor_locator(AutoMinorLocator(2))
ax_cool_full.grid(True, linestyle='--', alpha=0.6, zorder=0)
ax_cool_full.text(0.01, 1.02, 'a', transform=ax_cool_full.transAxes,
                  fontsize=fs['subtitle'], fontweight='black', va='bottom', ha='left')
ax_cool_full.set_title('Cooling: full range (0–0.2, >1.0 included)', fontsize=fs['legend'])

# --- Row 1: Heating ---
# Left: Full range
ax_heat_full.hist(heating_PLR_clipped, bins=bin_edges_full, density=True,
                  color=color_heating, alpha=alpha_hist, edgecolor='white', linewidth=0.5)
ax_heat_full.set_xlabel('Part load ratio [ - ]', fontsize=fs['label'], labelpad=pad['label'])
ax_heat_full.set_ylabel('Probability density [ - ]', fontsize=fs['label'], labelpad=pad['label'])
ax_heat_full.tick_params(axis='both', which='major', labelsize=fs['tick'], pad=pad['tick'])
ax_heat_full.set_xlim(0, 1.15)
ax_heat_full.set_ylim(0, 20)
ax_heat_full.set_xticks(np.arange(0, 1.2, 0.2))
ax_heat_full.xaxis.set_minor_locator(AutoMinorLocator(2))
ax_heat_full.yaxis.set_minor_locator(AutoMinorLocator(2))
ax_heat_full.grid(True, linestyle='--', alpha=0.6, zorder=0)
ax_heat_full.text(0.01, 1.02, 'b', transform=ax_heat_full.transAxes,
                  fontsize=fs['subtitle'], fontweight='black', va='bottom', ha='left')
ax_heat_full.set_title('Heating: full range (0–0.2, >1.0 included)', fontsize=fs['legend'])

# Optional: save comparison figure
# plt.savefig('../figure/PLR_histogram_comparison.png', dpi=600)
# plt.savefig('../figure/PLR_histogram_comparison.pdf', dpi=600)
dm.util.save_and_show(fig2)

# PLR 상하한 제한 없이 모든 부하 데이터

In [24]:
# =============================================================================
# PLR 전체 범위 (제한 없이) 히스토그램: 모든 실제 데이터 사용 (0 미만, 1.0 초과 포함)
# =============================================================================

# Full-range bins: 0부터 1.15까지 모든 구간 포함 (0.05 간격)
bin_edges_full = np.arange(0, 1.15, 0.05)

# 클리핑 없이 원본 데이터 그대로 사용
# (cooling_PLR, heating_PLR의 모든 값 포함)
fig2, axes = plt.subplots(2, 1, figsize=(dm.cm2in(17), dm.cm2in(10)))
plt.subplots_adjust(left=0.08, right=0.975, top=0.92, bottom=0.12, hspace=0.6)

ax_cool_full, ax_heat_full = axes[0], axes[1]

color_cooling = 'oc.blue6'
color_heating = 'oc.red6'
alpha_hist = 0.7

# --- Row 0: Cooling ---
ax_cool_full.hist(cooling_PLR, bins=bin_edges_full, density=True,
                  color=color_cooling, alpha=alpha_hist, edgecolor='white', linewidth=0.5)
ax_cool_full.set_xlabel('Part load ratio [ - ]', fontsize=fs['label'], labelpad=pad['label'])
ax_cool_full.set_ylabel('Probability density [ - ]', fontsize=fs['label'], labelpad=pad['label'])
ax_cool_full.tick_params(axis='both', which='major', labelsize=fs['tick'], pad=pad['tick'])
ax_cool_full.set_xlim(0, 1.15)
ax_cool_full.set_ylim(0, 20)
ax_cool_full.set_xticks(np.arange(0, 1.2, 0.2))
ax_cool_full.xaxis.set_minor_locator(AutoMinorLocator(2))
ax_cool_full.yaxis.set_minor_locator(AutoMinorLocator(2))
ax_cool_full.grid(True, linestyle='--', alpha=0.6, zorder=0)
ax_cool_full.text(0.01, 1.02, 'a', transform=ax_cool_full.transAxes,
                  fontsize=fs['subtitle'], fontweight='black', va='bottom', ha='left')
ax_cool_full.set_title('Cooling: All data, no PLR limits (shows <0.2, >1.0)', fontsize=fs['legend'])

# --- Row 1: Heating ---
ax_heat_full.hist(heating_PLR, bins=bin_edges_full, density=True,
                  color=color_heating, alpha=alpha_hist, edgecolor='white', linewidth=0.5)
ax_heat_full.set_xlabel('Part load ratio [ - ]', fontsize=fs['label'], labelpad=pad['label'])
ax_heat_full.set_ylabel('Probability density [ - ]', fontsize=fs['label'], labelpad=pad['label'])
ax_heat_full.tick_params(axis='both', which='major', labelsize=fs['tick'], pad=pad['tick'])
ax_heat_full.set_xlim(0, 1.15)
ax_heat_full.set_ylim(0, 20)
ax_heat_full.set_xticks(np.arange(0, 1.2, 0.2))
ax_heat_full.xaxis.set_minor_locator(AutoMinorLocator(2))
ax_heat_full.yaxis.set_minor_locator(AutoMinorLocator(2))
ax_heat_full.grid(True, linestyle='--', alpha=0.6, zorder=0)
ax_heat_full.text(0.01, 1.02, 'b', transform=ax_heat_full.transAxes,
                  fontsize=fs['subtitle'], fontweight='black', va='bottom', ha='left')
ax_heat_full.set_title('Heating: All data, no PLR limits (shows <0.2, >1.0)', fontsize=fs['legend'])

# Optional: save comparison figure
# plt.savefig('../figure/PLR_histogram_all_data.png', dpi=600)
# plt.savefig('../figure/PLR_histogram_all_data.pdf', dpi=600)
dm.util.save_and_show(fig2)